Reload Chroma ➜ contextualize follow-up questions with chat history (ChatAnthropic) ➜ retrieve with the standalone question

In [11]:
# chat model (ChatAnthropic)
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic

load_dotenv()  # loads ANTHROPIC_API_KEY from .env

llm = ChatAnthropic(
    model="claude-opus-5-5",
    max_tokens=4096,
    effort="low",
)

In [ ]:
# Build a contextualize-question step that uses chat history to rewrite follow-ups.
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Given a chat history and the latest user question, which might reference context in the "
     "chat history, rewrite it as a standalone question that can be understood without the chat "
     "history. Do NOT answer the question. If it is already standalone, return it unchanged. "
     "Reply with the question only."),
    MessagesPlaceholder("chat_history"),  # list[HumanMessage | AIMessage]
    ("human", "{question}"),
])

contextualize_question_chain = contextualize_prompt | llm | StrOutputParser()

# test
chat_history = [
    HumanMessage("What is the Transformer architecture?"),
    AIMessage("The Transformer is a sequence model built entirely on attention, with no recurrence or convolutions."),
]
follow_up = "How many attention heads does it use?"

standalone = contextualize_question_chain.invoke({"question": follow_up, "chat_history": chat_history})
print(f"follow-up:  {follow_up}\nstandalone: {standalone}\n")

In [ ]:
# Build a retriever (from Chroma) and an answer chain that grounds responses in retrieved text.
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.runnables import RunnablePassthrough

# retriever: reload the vector store persisted by notebook 01
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2") 
vectorstore = Chroma(persist_directory=".chroma/student-rag", embedding_function=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

question_retriever_chain = contextualize_question_chain | retriever  # question ➜ list[Document]

def join_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You answer questions about a collection of research papers. Use only the retrieved context "
     "below. If the context does not contain the answer, say you don't know instead of using outside "
     "knowledge.\n\n"
     "<context>\n{context}\n</context>"),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}"),
])

answer_chain = answer_prompt | llm | StrOutputParser()

# {"question", "chat_history"} ➜ add "context" (retrieved text) ➜ answer string
rag_chain = RunnablePassthrough | question_retriever_chain | join_docs | answer_chain

# test
answer = rag_chain.invoke({"question": follow_up, "chat_history": chat_history})
print(f"Q: {follow_up}\nA: {answer}")